In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=4)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="UpdraftArea",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Variable Loading Functions

def GetVarNames():
    return ['A_g','A_c']

def GetInputVariables(inputDataDirectory, timeString):

    ################################# DENSITY POTENTIAL TEMPERATURE
    
    varNames = GetVarNames()
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

In [ ]:
#Updraft Area Calculation Functions

def CalculateBinaryArea_V1(A, shapeApproximation="oval"): 
    """
    For each updraft point, calculate area as the true grid cell count
    of the 2D connected region it belongs to.

    NOT RECOMMENDED SINCE LARGE CONTIGUOUS REGIONS COUNTNED AS GIGANTIC UPDRAFT
    """
    [Nz, Ny, Nx] = A.shape
    updraftArea = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        labeled, n_features = label(slice2D)
        for i in range(1, n_features + 1):
            mask = labeled == i
            updraftArea[k][mask] = mask.sum()  # grid cell count, not x*y

    updraftArea *= ModelData.dx * ModelData.dy
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, xLength_out, yLength_out

def LabelSizesOfIndividualUpdrafts(array1D):
    """Label connected regions and return per-cell sizes."""
    labeled, _ = label(array1D)
    sizes = np.bincount(labeled.ravel())
    return labeled, sizes

def ApplyLengths(array1D, sizes,labeled):
    return np.where(array1D, sizes[labeled], np.nan)    

from scipy.ndimage import label
def CalculateBinaryLength(A, dimension, BC_x='open', BC_y='periodic'):
    """
    For each updraft point, calculate the length of the contiguous 
    updraft run it belongs to, along the specified dimension.
    """

    #Get Dimensional Information
    [Nz, Ny, Nx] = A.shape; length = np.full(A.shape, np.nan)
    [N, N_loop, BC] = (Nx, Ny, BC_x) if dimension == 'x' else (Ny, Nx, BC_y)

    #Looping
    for k in range(Nz): #Looping over all levels
        for n in range(N_loop): #Looping over each dimensional slice
            array1D = A[k, n, :] if dimension == 'x' else A[k, :, n] #Indexing corresponding 1d array
            if array1D.sum() == 0: continue
            
            if BC == 'periodic': #if boundary conditions are periodic, meaning updrafts continue to the other side past the boundary
                array1D = np.tile(array1D, 3)
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)
                lengthWhere = np.minimum(lengthWhere[N:2*N], N)
            elif BC == 'open': #if boundary conditions are open (e.g. radiative), meaning updrafts disappear at the boundary
                labeled, sizes = LabelSizesOfIndividualUpdrafts(array1D=array1D)
                lengthWhere = ApplyLengths(array1D=array1D, sizes=sizes, labeled=labeled)

            if dimension == 'x':
                length[k, n, :] = lengthWhere
            elif dimension == 'y':
                length[k, :, n] = lengthWhere
    return length

# def CalculateBinaryArea_V2(A,
#                         shapeApproximation="oval"):
#     """
#     For each updraft point, calculate the local cross-sectional area
#     as xLength * yLength.

#     NOT RECOMMENDED SINCE UPDRAFT AREA COULD BE LARGE, BUT VERY CLOSE TO EDGE OF CLOUD
#     """
#     xLength = CalculateBinaryLength(A, dimension='x')
#     yLength = CalculateBinaryLength(A, dimension='y')

#     #calculating updraft area
#     updraftArea = xLength * yLength
#     #converting from grid index to meters
#     updraftArea *= ModelData.dx * ModelData.dy
#     #converting to oval approximation
#     if shapeApproximation=="oval":
#         updraftArea *= (np.pi / 4)
    
#     return updraftArea, xLength,yLength

def CalculateBinaryArea_V3(A, shapeApproximation="oval"):
    """
    For each updraft point, calculate area as mean_x_length * mean_y_length
    for the 2D connected region it belongs to.

    #RECOMMENDED
    """
    [Nz, Ny, Nx] = A.shape
    area = np.full(A.shape, np.nan)
    xLength_out = np.full(A.shape, np.nan)
    yLength_out = np.full(A.shape, np.nan)

    # Get per-point x and y lengths
    xLength = CalculateBinaryLength(A, dimension='x')
    yLength = CalculateBinaryLength(A, dimension='y')

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue

        labeled, n_features = label(slice2D)

        for i in range(1, n_features + 1):
            mask = labeled == i
            xLength_out[k][mask] = np.nanmean(xLength[k][mask])
            yLength_out[k][mask] = np.nanmean(yLength[k][mask])

    #calculating updraft area
    updraftArea = xLength_out * yLength_out
    #converting from grid index to meters
    updraftArea *= ModelData.dx * ModelData.dy
    #converting to oval approximation
    if shapeApproximation == "oval":
        updraftArea *= (np.pi / 4)

    return updraftArea, xLength_out, yLength_out

from scipy.ndimage import distance_transform_edt
def CalculateEuclideanDistanceFromNearestEdge(A,dy=ModelData.dy,dx=ModelData.dx):
    """
    For each updraft point, calculate the Euclidean distance to the 
    nearest non-updraft cell.
    """
    [Nz, Ny, Nx] = A.shape
    distance = np.full(A.shape, np.nan)

    for k in range(Nz):
        slice2D = A[k, :, :]
        if slice2D.sum() == 0: continue
        # sampling gives physical distance in meters
        distance[k] = np.where(slice2D, distance_transform_edt(slice2D, sampling=(dy,dx)), np.nan)

    return distance

In [ ]:
#Running Functions

def VariableCalculation(inputDictionary):
    varNames = GetVarNames()
    [A_g,A_c] = (inputDictionary[k] for k in varNames)

    [updraftArea_g,_,_] = CalculateBinaryArea_V3(A_g)
    [updraftArea_c,_,_] = CalculateBinaryArea_V3(A_c)

    updraftEdgeDistance_g = CalculateEuclideanDistanceFromNearestEdge(A_g)
    updraftEdgeDistance_c = CalculateEuclideanDistanceFromNearestEdge(A_c)
    
    outputDictionary={'updraftArea_g': updraftArea_g,
                      'updraftArea_c': updraftArea_c,
                      'updraftEdgeDistance_g': updraftEdgeDistance_g,
                      'updraftEdgeDistance_c': updraftEdgeDistance_c}
    
    return outputDictionary

def RunCode():
    for t in tqdm(loop_elements, total=len(loop_elements)):
        if np.mod(t,1)==0: print(f'Current time {t}')    
    
        #loading input variables
        timeString = ModelData.timeStrings[t]
        inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)
    
        #calculating variables
        outputDictionary = VariableCalculation(inputDictionary)
        
        #outputting
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [ ]:
####################################
#RUNNING

In [ ]:
#CALCULATING AND APPENDING TO DATA EACH TIMESTEP
RunCode()

In [ ]:
####################################
#TESTING

In [ ]:
# #testing CalculateUpdraftArea_V2

# slice2D = [[1,0,1,1,0],
#        [0,1,1,1,0],
#        [1,1,0,0,0]]
# labeled, n_features = label(slice2D)
# labeled

In [ ]:
# #testing plotting a single cloud

# def TestPlot_1():

#     fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
    
#     plotData = [(xLength_c[z], "xLength (index)"), (yLength_c[z], "yLength (index)"), (updraftArea_c[z] / 1e6, "Updraft Area (km²)")]
    
#     for i, (axis, (array, title)) in enumerate(zip(axes, plotData)):
    
#         if i < 2:
#             levels = np.arange(np.nanmin(array), np.nanmax(array) + 1, 1)
#         else:
#             levels = np.linspace(np.nanmin(array), np.nanmax(array), 14)
    
#         contour = axis.contourf(array, levels=levels)
#         fig.colorbar(contour, ax=axis)
    
#         axis.set_title(title)
#         axis.set_xlim(268, 285); axis.set_ylim(70, 85)
#         axis.grid()
#         axis.xaxis.set_major_locator(plt.MultipleLocator(1)); axis.yaxis.set_major_locator(plt.MultipleLocator(1))
#         axis.tick_params(axis="x", rotation=90)

# def TestPlot_2():

#     fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
    
#     for axis, data, xlabel in zip(
#         axes,
#         [xLength_c, yLength_c, updraftArea_c / 1e6],
#         ['LengthX_c (km)', 'LengthY_c (km)', 'Area_c (km²)']
#     ):
#         axis.plot(np.nanmean(data, axis=(1, 2)), ModelData.zh)
#         axis.set_xlabel(xlabel)
    
#     axes[0].set_ylabel('z (km)')
    
#     plt.tight_layout()

# #Example for Simulation One
# t=120
# z=20
# timeString = ModelData.timeStrings[t]
# inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)
# A_c=inputDictionary['A_c']*1

# [updraftArea_c,xLength_c,yLength_c] = CalculateBinaryArea_V3(A_c)
# TestPlot_1()
# TestPlot_2()

In [ ]:
# #Example for Simulation One
# t=120
# z=5
# timeString = ModelData.timeStrings[t]
# inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)

# A_c=inputDictionary['A_g']*1
# [updraftArea_c,xLength_c,yLength_c] = CalculateBinaryArea_V2(A_c)
# TestPlot_1()
# TestPlot_2()

In [ ]:
# #testing full domain plot comparing both updraft area versions

# def TestPlot_1():

#     fig, axes = plt.subplots(
#         2, 2,
#         figsize=(10, 8),
#         sharex=True,
#         sharey=True
#     )

#     variables = [
#         updraftArea_c_1[z] / 1e6,
#         updraftArea_c_2[z] / 1e6,
#         updraftArea_c_3[z] / 1e6,
#     ]

#     titles = [
#         "Area (v1)",
#         "Area (v2)",
#         "Area (v3)"
#     ]

#     levels = np.linspace(
#         np.nanmin(variables),
#         np.nanmax(variables),
#         15
#     )

#     for axis, variable, title in zip(axes.ravel(), variables, titles):

#         im = axis.contourf(variable, levels=levels)
#         fig.colorbar(im, ax=axis)

#         axis.set_xlim(100, 400)
#         axis.set_ylim(50, 150)
#         axis.set_title(title)

#     # Hide unused panel
#     axes[1, 1].axis("off")

#     # Axis labels
#     axes[1, 0].set_xlabel("X (index)")
#     axes[1, 1].set_xlabel("X (index)")  # hidden anyway

#     axes[0, 0].set_ylabel("Y (index)")
#     axes[1, 0].set_ylabel("Y (index)")

#     plt.subplots_adjust(wspace=0.03, hspace=0.2)

# def TestPlot_2():

#     fig, axis = plt.subplots(figsize=(5, 8))

#     area1 = np.nanmean(updraftArea_c_1 / 1e6, axis=(1, 2))
#     area2 = np.nanmean(updraftArea_c_2 / 1e6, axis=(1, 2))
#     area3 = np.nanmean(updraftArea_c_3 / 1e6, axis=(1, 2))

#     axis.plot(area1, ModelData.zh, label="Area (v1)")
#     axis.plot(area2, ModelData.zh, label="Area (v2)")
#     axis.plot(area3, ModelData.zh, label="Area (v3)")

#     axis.set_xlabel("Mean Area (km²)")
#     axis.set_ylabel("Height (km)")
#     axis.legend()

#     plt.tight_layout()


# #Example for Simulation One
# t=120
# z=3
# timeString = ModelData.timeStrings[t]
# inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)
# A_c=inputDictionary['A_g']*1

# [updraftArea_c_1,_,_] = CalculateBinaryArea_V1(A_c)
# [updraftArea_c_2,_,_] = CalculateBinaryArea_V2(A_c)
# [updraftArea_c_3,_,_] = CalculateBinaryArea_V3(A_c)


# TestPlot_1()
# TestPlot_2()

# # updraftArea_c_3=updraftArea_c_2
# # TestPlot_2()

In [ ]:
# #Testing CalculateDistanceFromEdge

# t=120
# z=20
# timeString = ModelData.timeStrings[t]
# inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)

# A_c=inputDictionary['A_c']*1

# a = CalculateEuclideanDistanceFromNearestEdge(A_c)[z]

# def TestPlot():
#     fig, axis = plt.subplots(figsize=(8, 6))
    
#     im = axis.contourf(a)
#     fig.colorbar(im, ax=axis)
    
#     axis.set_xlim(268, 285)
#     axis.set_ylim(70, 85)
    
#     axis.grid()
    
#     axis.xaxis.set_major_locator(plt.MultipleLocator(1))
#     axis.yaxis.set_major_locator(plt.MultipleLocator(1))
    
#     axis.tick_params(axis="x", rotation=90)

#     axis.set_title(f"Euclidean Distance from Nearest Updraft Edge (m)")
    
# TestPlot()